In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")
from src.data_preprocessing import load_data, remove_unwanted_columns, convert_to_datetime, sort_by_date_and_productId
from src.split_data import split_data, split_features_and_target
from pipeline.train_model import run_pipeline
from pipeline.features import add_engineered_features

In [2]:
df = load_data("../data/raw/stocksense_dataset.csv")
df.head()


,date,product_id,product_name,category,price,quantity_sold,stock_before,stock_after,reorder_level
0,2024-01-01,P001,Rice,Grains,1502,13,121,108,30
1,2024-01-01,P002,Milk,Dairy,1288,11,124,113,30
2,2024-01-01,P003,Bread,Bakery,902,14,180,166,30
3,2024-01-01,P004,Sugar,Groceries,1049,16,70,54,30
4,2024-01-01,P005,Eggs,Protein,2060,8,138,130,30


In [3]:
df = remove_unwanted_columns(df, ["category", "stock_before", "reorder_level"])
df.head()

,date,product_id,product_name,price,quantity_sold,stock_after
0,2024-01-01,P001,Rice,1502,13,108
1,2024-01-01,P002,Milk,1288,11,113
2,2024-01-01,P003,Bread,902,14,166
3,2024-01-01,P004,Sugar,1049,16,54
4,2024-01-01,P005,Eggs,2060,8,130


In [4]:
df = convert_to_datetime(df, "date")


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           900 non-null    datetime64[ns]
 1   product_id     900 non-null    object        
 2   product_name   900 non-null    object        
 3   price          900 non-null    int64         
 4   quantity_sold  900 non-null    int64         
 5   stock_after    900 non-null    int64         
dtypes: datetime64[ns](1), int64(3), object(2)
memory usage: 42.3+ KB


In [6]:
df.isnull().sum()

date             0
product_id       0
product_name     0
price            0
quantity_sold    0
stock_after      0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df = add_engineered_features(df)

In [9]:
df.head()

,product_id,price,quantity_sold,stock_after,quantity_sold_lag1,Day_of_week
5,P001,1448,10,54,13.0,1
10,P001,1574,7,92,10.0,2
15,P001,1612,9,93,7.0,3
20,P001,1427,15,67,9.0,4
25,P001,1604,8,93,15.0,5


In [10]:
train_df, val_df, test_df = split_data(df)

In [11]:
train_df.head()

,product_id,price,quantity_sold,stock_after,quantity_sold_lag1,Day_of_week
5,P001,1448,10,54,13.0,1
10,P001,1574,7,92,10.0,2
15,P001,1612,9,93,7.0,3
20,P001,1427,15,67,9.0,4
25,P001,1604,8,93,15.0,5


In [12]:
X_train, y_train = split_features_and_target(train_df)
X_val, y_val = split_features_and_target(val_df)
X_test, y_test =split_features_and_target(test_df)

In [13]:
X_train.head()

,product_id,price,stock_after,quantity_sold_lag1,Day_of_week
5,P001,1448,54,13.0,1
10,P001,1574,92,10.0,2
15,P001,1612,93,7.0,3
20,P001,1427,67,9.0,4
25,P001,1604,93,15.0,5


In [14]:
X_train.head()

,product_id,price,stock_after,quantity_sold_lag1,Day_of_week
5,P001,1448,54,13.0,1
10,P001,1574,92,10.0,2
15,P001,1612,93,7.0,3
20,P001,1427,67,9.0,4
25,P001,1604,93,15.0,5


In [15]:
pipeline = run_pipeline(X_train, y_train)

In [16]:
pipeline.score(X_val, y_val)

0.061708810798414504

In [17]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred = pipeline.predict(X_val)

mse = mean_squared_error(y_val, y_pred)
mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"Mean Absolute Error: {mae}")
print(f"R² Score: {r2}")

Mean Squared Error: 15.390567352036344
Mean Absolute Error: 3.1412894325655873
R² Score: 0.061708810798414504
